# 01 — Data Exploration

Load FabCon 2026 sessions from the Obsidian vault and explore the dataset.

**Topics:** session counts, track distribution, level breakdown, speaker analysis, timeline.

In [ ]:
import sys
sys.path.insert(0, '../../src/eda')  # add eda package to path

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from eda.data.loader import load_vault
from eda.data.schema import ALL_TRACKS, DAY_ORDER

VAULT = '../../'

df = load_vault(VAULT)
print(f'Loaded {len(df)} sessions')
print(f'Columns: {list(df.columns)}')

In [ ]:
df.dtypes

In [ ]:
df[['title','day','track','level','conference','status','interest','duration']].head(10)

## Conference & Day Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Conference pie
conf_counts = df['conference'].value_counts()
axes[0].pie(conf_counts, labels=conf_counts.index, autopct='%1.0f%%',
            colors=['#0078D4','#00B7C3'])
axes[0].set_title('Sessions by Conference')

# Day bar
day_order = [d for d in DAY_ORDER if d in df['day'].values]
day_counts = df['day'].value_counts().reindex(day_order, fill_value=0)
axes[1].bar(day_counts.index, day_counts.values, color='steelblue')
axes[1].set_title('Sessions by Day')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')
plt.setp(axes[1].get_xticklabels(), rotation=20)

plt.tight_layout()
plt.show()

## Track Distribution

In [ ]:
track_counts = df['track'].value_counts()
fig, ax = plt.subplots(figsize=(12, 7))
track_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Session Count')
ax.set_title('Sessions per Track')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print(track_counts.to_string())

## Technical Level Breakdown

In [ ]:
level_map = {100: 'L100 Business', 200: 'L200 Feature', 300: 'L300 Technical', 400: 'L400 Deep Tech'}
level_counts = df['level'].value_counts().sort_index()
labels = [level_map.get(int(l), str(l)) for l in level_counts.index if pd.notna(l)]
values = level_counts.values

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=['#4CAF50','#2196F3','#FF9800','#F44336'])
ax.bar_label(bars)
ax.set_title('Sessions by Technical Level')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Top Speakers

In [ ]:
# Explode the speakers list column
speakers_series = df['speakers'].explode().dropna()
speakers_series = speakers_series[speakers_series.astype(str).str.strip() != '']
top_speakers = speakers_series.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top_speakers[::-1].plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Number of Sessions')
ax.set_title('Top 20 Speakers by Session Count')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Session Type & Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Session type
type_counts = df['session_type'].value_counts().head(10)
type_counts.plot(kind='barh', ax=axes[0], color='purple')
axes[0].set_title('Top Session Types')
axes[0].grid(axis='x', alpha=0.3)

# Duration
if 'duration' in df.columns and df['duration'].notna().any():
    df['duration'].dropna().astype(int).value_counts().sort_index().plot(
        kind='bar', ax=axes[1], color='orange')
    axes[1].set_title('Session Duration Distribution (minutes)')
    axes[1].set_xlabel('Duration (min)')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Missing Value Analysis

In [ ]:
key_cols = ['title','track','level','day','room','speakers','description','interest','status']
missing = df[key_cols].isnull().sum()
empty_str = df[key_cols].applymap(lambda x: x == '' if isinstance(x, str) else False).sum()
summary = pd.DataFrame({'missing': missing, 'empty': empty_str})
summary['pct_missing'] = (summary['missing'] + summary['empty']) / len(df) * 100
print(summary.sort_values('pct_missing', ascending=False).to_string())

## Interest & Status Summary (User Planning)

In [ ]:
print('Status distribution:')
print(df['status'].value_counts().to_string())
print('\nInterest distribution (1-5):')
print(df['interest'].value_counts().sort_index().to_string())